In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [6]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [7]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [8]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [9]:
print(response.output_text)

The provided context does not contain specific instructions on how to create a dashboard in Evidently. Therefore, I cannot answer your question based on the available information.


In [10]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [11]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [12]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [13]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_6cdDf8o7n0wqPWW74D6YqZx1', name='search', type='function_call', id='fc_0d45a6df214a017d006acb8a03999c87d1af692dc7a8eb1656', async_=None, caller=None, namespace=None, status='completed')

In [14]:
messages.append(tool_call)

In [15]:
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [16]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidently'}

In [17]:
search_results = search(query='create dashboard in Evidently')

In [18]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [19]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [20]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_6cdDf8o7n0wqPWW74D6YqZx1', name='search', type='function_call', id='fc_0d45a6df214a017d006acb8a03999c87d1af692dc7a8eb1656', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_6cdDf8o7n0wqPWW74D6YqZx1',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation 

In [21]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3685

In [22]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [23]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3909

In [24]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, you'll need to follow these steps:

1. **Create a Project**: Start by creating a project in Evidently. You can create an account on [Evidently Cloud](https://app.evidently.cloud/signup) if you haven't done so yet.

2. **Run an Evaluation**: You need to run an evaluation and save at least one report to the project as a data source.

3. **Access the Dashboard**: Once you have reports saved, you can access the dashboard associated with your project. The dashboard will be empty at first.

4. **Add Panels**: Panels are visual elements that display specific values. You can add panels in two ways:
   - Using the **Python API**: Define and customize the dashboard programmatically.
   - Through the **User Interface (UI)**: If you're using Evidently Cloud or Enterprise, you can add panels directly from the interface.

   To add a panel using the API, you need to specify the following:
   - **Value**: Choose a metric to plot.
   - **Parameters**: Such as title,

In [25]:
##Agentic Loop

In [26]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [27]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [28]:
question = "How do I create a dahsbord in Evidently?"

In [29]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [30]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [31]:
message_history.extend(response.output)

In [32]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_nW2ApH280rGgyFlzogkXJG1R', name='search', type='function_call', id='fc_054fa419d9660a89006acb8a1c0d0887d1b725c4c9fcc3ad46', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard features"}', call_id='call_OnWI2qVXxYXvancPW0aR07PC', name='search', type='function_call', id='fc_05

In [33]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard features"})...
executing search({"query":"dashboard setup"})...


In [34]:
len(message_history)

8

In [35]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

12147

In [36]:
print(response.output_text)

To create a dashboard in Evidently, you can follow these steps, depending on whether you want to use the user interface or the Python API.

### Using the User Interface

1. **Enter Edit Mode**:
   - On your dashboard, click on the "Edit" mode in the top right corner.

2. **Add Tabs**:
   - Click on the plus sign to "Add Tab" on the left.
   - You can create a custom tab or use pre-built templates that organize panels for you.

3. **Add Panels**:
   - Click the "Add Panel" button.
   - Follow the prompts to configure your panel:
     - Select metrics to display.
     - Optionally filter by tags to narrow down the data source.
   - Use the preview to validate your configuration and then save the panel to the desired tab.

### Using the Python API

1. **Connect to Your Project**:
   - Ensure you are connected to Evidently Cloud and have created a project.

2. **Adding Tabs**:
   - You can add tabs programmatically using:
     ```python
     project.dashboard.add_tab("Tab Name")
     ```



In [37]:
## Agentic Loop

In [38]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [39]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To assist you in creating a dashboard, I will first search for general guidelines or documentation related to dashboard creation. This will help establish a foundational understanding of the process and any prerequisites that may be necessary.

Let's begin with the initial search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The first search revealed several key aspects of creating a dashboard, including the ability to add panels and tabs through the user interface or the Python API. It highlights that dashboards are specific to each project and that the first step involves generating reports which serve as data sources for the panels. It also mentions creating and managing dashboards as code, which is suitable for more advanced users.

Next, I'll explore further by searching for specific steps or examples on adding panels and tabs using both the UI and the Python API. This will provide a clear, step-by-step ap

In [40]:
#adding more toolS

In [41]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [42]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [43]:
#make tools work together

In [44]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To provide you with the best guidance on creating a dashboard, I'll first search for general information regarding dashboard creation. This will help establish a foundational understanding of the steps involved and any relevant features associated with dashboard functionality.

I'll perform the first search now.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: From the first search, I found information on how to create panels and manage dashboards. It appears that you can either use the UI or the Python API to add panels and manage dashboard tabs. The content also mentions that you need to have reports saved in your project to populate the dashboard effectively.

For the second iteration, I'll analyze this information to better understand panel and dashboard management. I will perform two additional searches: one focused on the user interface method for creating dashboards, and another on the specific features and 

In [45]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [46]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"docs/platform/dashboard_creation.mdx","title":"Creating Dashboards","description":"Guide on creating dashboards using the user interface and Python API.","content":"### Creating Dashboards:\n1. **Using the User Interface**:\n   - Enter \"Edit\" mode on your dashboard, click the plus sign to add a new tab, then select or create a custom tab.\n   - Adding panels is straightforward; just click \"Add Panel,\" follow prompts to configure it (selecting metrics and visual types), and then save it to a specified tab.\n\n2. **Using the Python API**:\n   - You can manage your dashboard programmatically using methods to add tabs and panels. For instance, to create a new tab:\n     ```python\n     project.dashboard.add_tab(\"My New Tab\")\n     ```\n   - Adding panels can be done with the `add_panel` method, where you can specify attributes like title, size, and plot type:\n     ```python\n     project.dashboard.add_panel(\n         Dashboar

In [47]:
index.docs[-1]

{'start': 0,
 'content': '### Overview of Dashboards:\n- A Dashboard provides a clear view of your AI application performance, tracking evaluation results across experiments and monitoring live production quality.\n- Each Project has its own Dashboard, which starts empty and is populated by reports.\n- Dashboards can contain multiple tabs, allowing you to organize panels logically.',
 'title': 'Overview of Dashboards',
 'description': 'Introduction to dashboards and their functionality.',
 'filename': 'docs/platform/dashboard_overview.mdx'}

In [48]:
#The QA Loop

In [49]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

You: agents for dashboards


iteraration number 1...
executing search({"query":"agents for dashboards"})...

iteraration number 2...
ASSISTANT: The first search provided an overview of dashboards and guide on how to create them, but there was no direct mention of agents specifically associated with dashboards. Since "agents" may refer to various roles or functionalities related to dashboards, further investigation is needed to explore specific functionalities that might be categorized under agents or monitoring tools related to dashboards.

I'll perform two more searches: one focusing on "monitoring dashboards" to see if there are any agents used for monitoring purposes, and another on "dashboard functionalities" to check for any features that could categorize under agent-like operations. This should help clarify if there are agents or similar functionalities for dashboards. 

Let's go ahead and conduct these two searches.
executing search({"query":"monitoring dashboards"})...
executing search({"query":"dashboard 

You: stop


In [50]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

In [51]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [52]:
messages = agent.loop('evidently adshboards')

iteraration number 0...
ASSISTANT: I want to perform a search for "adshboards" to find relevant documentation about this feature. Dashboards are typically a key component in any analytics or data visualization tool, and understanding how to create or utilize them effectively is crucial for users.

Let's proceed with the search.
executing search({"query":"adshboards"})...

iteraration number 1...
ASSISTANT: It seems that there are no direct results for "adshboards" in the documentation. This might be due to a typo or misunderstanding of the term; it is possible that the search term should be "dashboards."

To verify this, I will search for "dashboards" to see if relevant information can be found under that term. This approach will help ensure that I cover all possible terminology the documentation may utilize. Let's execute this search.
executing search({"query":"dashboards"})...

iteraration number 2...
ASSISTANT: The search for "dashboards" yielded several relevant results. Here's a s

In [54]:
agent.qna()

You: stop


In [59]:
# toykit as a stepping stone

In [55]:
!uv add toyaikit

Resolved 142 packages in 1ms
Checked 138 packages in 1ms


In [56]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [57]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [58]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

In [59]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

from toyaikit.tools import Tools

from toyaikit.tools import Tools

In [60]:
from toyaikit.tools import Tools

In [61]:
# creating agent

In [62]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [63]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

In [64]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

In [65]:
# Running the Agent

In [66]:
result = agent.loop('how can I create evidently dahsbord?')

In [67]:
print(result.last_message)

To create a dashboard, you can follow these general steps:

1. **Getting Started**: Ensure you're connected to the Cloud or have the required access to the OSS or Enterprise versions. You need to create a project and save at least one report to populate your dashboard.

2. **Adding Tabs**:
   - Enter the "Edit" mode on your dashboard.
   - Click the "+" icon to add a new tab. You can choose to create a custom tab or use pre-built templates.

3. **Adding Panels**:
   - Panels are the visual elements that display your metrics. You can add them in two ways: through a Python API or directly from the User Interface (UI).
   - To add a new panel via the UI:
     - Click on the "Add Panel" button.
     - Select the metrics you want to display and configure the panel type (e.g., line chart, bar chart, counter).
     - Click "Save" to finalize your panel.

4. **Using the Python API**: If you prefer coding, you can also add panels using the provided Python scripts.
   - For example, to add a tex

In [68]:
result.cost.total_cost

Decimal('0.0027720')

In [69]:
# adding visability

In [70]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [71]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [72]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [73]:
#continuing the conversation

In [74]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [75]:
# Q and A Loop

In [76]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [77]:
result = agent.run()

You: dahsboard what


-> Response received


-> Response received


-> Response received


You: stop


Chat ended.


In [78]:
result.cost.total_cost

Decimal('0.00298395')

In [79]:
#organizing tools in a class

In [80]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [81]:
def search(query: str) -> list[dict]:
    """
    Search the documentation database for relevant results based on a query string.

    Args:
        query (str): The search query to look up in the index.

    Returns:
        list[dict]: A list of search result objects returned by the index.
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return results

def add_entry(filename, title, description, content):
    """
    Add a new documentation entry to the index.

    Args:
        filename (str): The source filename associated with the entry.
        title (str): The title of the documentation entry.
        description (str): A short description summarizing the entry.
        content (str): The full content of the documentation entry.

    Returns:
        str: A status message indicating the result of the operation ("OK").
    """
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [82]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    list[dict]: A list of search result objects returned by the index.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filenam

In [83]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [84]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [85]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [86]:
class SearchIndexTools:
    def __init__(self, index):
        self.index = index

    def search(self, query: str) -> list[dict]:
        """
        Search the documentation database for relevant results based on a query string.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            list[dict]: A list of search result objects returned by the index.
        """
        results = self.index.search(
            query=query,
            num_results=5
        )
        return results

    def add_entry(self, filename, title, description, content):
        """
        Add a new documentation entry to the index.

        Args:
            filename (str): The source filename associated with the entry.
            title (str): The title of the documentation entry.
            description (str): A short description summarizing the entry.
            content (str): The full content of the documentation entry.

        Returns:
            str: A status message indicating the result of the operation ("OK").
        """
        entry = {
            'start': 0,
            'content': content,
            'title': title,
            'description': description,
            'filename': filename,
        }
        self.index.append(entry)
        return "OK"

In [87]:
search_tools_object = SearchIndexTools(index)
results = search_tools_object.search('dashboard')
results

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [88]:
agent_tools = Tools()
agent_tools.add_tool(search_tools_object.search)
agent_tools.add_tool(search_tools_object.add_entry)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    list[dict]: A list of search result objects returned by the index.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filenam

In [89]:
agent_tools = Tools()
agent_tools.add_tools(search_tools_object)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename (str): The source filename associated with the entry.\n    title (str): The title of the documentation entry.\n    description (str): A short description summarizing the entry.\n    content (str): The full content of the documentation entry.\n\nReturns:\n    str: A status message indicating the result of the operation ("OK").',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title': {'type': 'string', 'description': 'title parameter'},
    'description': {'type': 'string', 'description': 'description parameter'},
    'content': {'type': 'string', 'description': 'content parameter'}},
   'required': ['filename', 'title', 'description', 'content'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database f

In [90]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [92]:
import search_tools

search_tools_object = search_tools.SearchIndexTools(index)
result = search_tools_object.search('dashboard')
result

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [93]:
##Beyond Rag

In [95]:
!uv add "minsearch==0.0.9"


Resolved 141 packages in 2.71s                                       
Prepared 2 packages in 102ms                                             
Uninstalled 2 packages in 12ms
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 15msamp-codespaces==0.1.0 (from file
 ~ ai-engineering-buildcamp-codespaces==0.1.0 (from file:///workspaces/ai-engineering-buildcamp-codespaces)
 - minsearch==0.2.0
 + minsearch==0.0.9


In [96]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

In [101]:
query = 'how do I use evidently to monitor my machine learning models?'

In [102]:
search_results = index.search(query)

In [103]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [104]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords)
)

In [105]:
highlighter.highlight(query, search_results)

[{'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'content': {'matches': ['...downloading and using ML **models** from HuggingFace. This l...',
    '...criteria from the source **model**, e.g. classify texts by ...',
    '.... There are:\n\n* Ready-to-**use** descriptors that wrap a ...'],
   'total_matches': 33},
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'title': 'Evidently Cloud',
  'description': 'How to set up Evidently Cloud account.',
  'icon': 'cloud',
  'content': {'matches': ['... and Evidently Cloud v1, **use** `from evidently.ui.works...'],
   'total_matches': 1},
  'filename': 'docs/setup/cloud.mdx'},
 {'title': 'Why Evidently?',
  'description': 'Why choose Evidently.',
  'content': {'matches': ['...ucts: whether it’s an ML **model**, an LLM app, or a comple...',
    '...workflow.\n\nOur tools are **model**-, framework-, and applic...',
    '.... It’s up to you.\xa0\n- Add **monitoring** as you are re

In [106]:
file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

In [107]:
file_index["docs/setup/self-hosting.mdx"]

'<Note>\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform (as described on this page), or use the Evidently library with other tools.\n</Note>\n\nIn addition to using Evidently Python library, you can self-host the UI Service to get a monitoring Dashboard and organize the results of your evaluations. This is optional: you can also view evaluation results in Python or export to JSON or HTML.\n\nTo get a self-hostable Dashboard, you must:\n\n- Create a Workspace (local or remote) to store your data.\n- Launch the UI service.\n\n## 1. Create a Workspace\n\nOnce you [install Evidently](/docs/setup/installation), you will need to create a `workspace`. This designates a remote or local directory where you will store the evaluation results (JSON Reports called `snapshots`), traces or datasets. The UI Service will read the data from this source.\n\nAs a storage backend, Evidently supports:\n\n- a file system\n- any SQL-like da

In [108]:
from typing import Any, Dict, List

class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """
    def __init__(self, index: Any, highlighter: Any, file_index: Dict[str, str]) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [109]:
search_tools = SearchTools(index, highlighter, file_index)

In [110]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [111]:
#Running the Agent

In [112]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': "Retrieve a file's contents by filename.\nArgs:\n    filename (str): The filename of the file to retrieve.\nReturns:\n    str: The file contents if found, otherwise an error message.",
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the index for results matching a query and highlight them.\nArgs:\n    query (str): The search query to look up in the index.\nReturns:\n    List[Dict[str, Any]]: A list of highlighted search result objects.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [113]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [114]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [115]:
result = agent.loop(
    query + " show me the code",
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [116]:
tools = [search_tools.search, search_tools.get_file]

In [117]:
from toyaikit.tools import get_instance_methods

tools = get_instance_methods(search_tools)

In [121]:
! uv add pydantic_ai


Resolved 201 packages in 3.34s                                       
Prepared 59 packages in 772ms                                            
Uninstalled 1 package in 18ms
░░░░░░░░░░░░░░░░░░░░ [0/59] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 59 packages in 2.05s4.0                           
 ~ ai-engineering-buildcamp-codespaces==0.1.0 (from file:///workspaces/ai-engineering-buildcamp-codespaces)
 + aiofile==3.12.3
 + argcomplete==3.7.2
 + authlib==1.8.0
 + beartype==0.23.1
 + cachetools==7.2.1
 + caio==0.12.9
 + distro==1.9.0
 + dnspython==2.9.0
 + email-validator==2.3.0
 + exceptiongroup==1.3.1
 + fastmcp-slim==4.1.0
 + google-auth==2.61.0
 

In [122]:
tools = [search_tools.search, search_tools.get_file]

In [123]:
from toyaikit.tools import get_instance_methods

tools = get_instance_methods(search_tools)

In [124]:
from pydantic_ai import Agent

In [125]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [126]:
query = 'how do I use evidently to monitor my machine learning models?'

In [127]:
result = await search_agent.run(query)

In [134]:
result.output

'In the retrieved documents, I found key information about how to use Evidently to monitor machine learning models. The main methods for monitoring include:\n\n1. **Batch Monitoring Jobs**: This is suited for batch ML pipelines and regression testing. You can create an evaluation pipeline in your infrastructure that runs at regular intervals or is triggered by new data. The pipeline will calculate metrics and store results, which can then be visualized on a dashboard. This approach provides full control over the evaluation process, essential for processing data in batches.\n\n2. **Tracing with Scheduled Evaluations**: This method is particularly useful for applications powered by large language models (LLMs). It involves instrumenting your application for data capture, storing raw data, and scheduling evaluations to run automatically. This setup not only facilitates complex data management but also allows for easy reruns of evaluations and no-code management through the UI.\n\nEvidentl

In [129]:
messages = result.all_messages()

In [133]:
result.usage()

TypeError: 'RunUsage' object is not callable